# Урок 16 — Наслідування, MRO та Композиція

**Модуль 2 · Python Intermediate · Автор: Viktor Nikoriak**

---

## Про що цей урок?

Більшість початківців думають, що наслідування — це спосіб "скопіювати" код з одного класу в інший.
Це хибна ментальна модель.

> **Головна ідея уроку:**  
> **Наслідування — це НЕ копіювання. Це система пошуку методів (method lookup) через ієрархію класів.**

Ми побачимо, як Python вирішує який метод виконати, розберемо алгоритм лінеаризації C3 і з'ясуємо, чому `super()` робить зовсім не те, що більшість думає.

---

## Зміст уроку

| # | Тема | Ключова ідея |
|---|------|--------------|
| 1 | Базове наслідування як шлях пошуку | MRO — лінійна черга класів |
| 2 | Перевизначення та `super()` | `super()` — не "батько", а "наступний у MRO" |
| 3 | Проблема ромба | C3 Linearization вирішує конфлікт |
| 4 | Кооперативне множинне наслідування | Найбільший WTF у Python |
| 5 | Пастки множинного наслідування | Обрив ланцюга та несумісні аргументи |
| 6 | Спадкування vs Композиція | "is-a" vs "has-a" — вибір архітектора |
| 7 | Чому наслідування порушує інкапсуляцію | Теоретична база дизайну |

---

## Частина 1: Базове наслідування як шлях пошуку

Коли ми пишемо `class Child(Parent):`, Python **нічого не копіює** з `Parent` до `Child`.
Замість цього він будує **MRO (Method Resolution Order)** — лінійний список класів, який є "шляхом пошуку".

### Передбачення

Клас `Child` порожній, але успадковує `Parent`. Коли ми викликаємо `c.speak()`, Python не знаходить цей метод у `Child`.
Що відбудеться? Чи виникне помилка?

In [ ]:
class Parent:
    def speak(self):
        print("Я — Parent")

# Порожній клас, який успадковує Parent
class Child(Parent):
    pass

c = Child()
c.speak()  # Що відбудеться?

# Перегляньмо шлях пошуку методів
print("MRO (шлях пошуку):", Child.mro())

**Пояснення:**

Помилки немає — метод знайдено через MRO.

Коли ви викликаєте `c.speak()`, Python виконує такий алгоритм:

```
1. Шукаємо 'speak' у словнику Child.__dict__   → не знайдено
2. Шукаємо 'speak' у словнику Parent.__dict__  → знайдено! Виконуємо.
3. Якщо б не знайшли і тут — перейшли б до object.__dict__
```

MRO для `Child` виглядає так: `[Child, Parent, object]`

> **Ключова ідея:**  
> Наслідування — це не копіювання коду. Це посилання на ланцюжок класів, якими Python буде рухатися під час пошуку.

Якщо метод перевизначено (overriding) у класі-нащадку, Python знаходить його на першому кроці, і пошук припиняється.

In [ ]:
class User:
    def login(self):
        print("login")

class Admin(User):
    pass

admin = Admin()
admin.login()

---

## Частина 2: Перевизначення методів та `super()`

**Перевизначення (overriding)** — коли клас-нащадок замінює поведінку методу батька.
Але що, якщо ми хочемо розширити батьківський метод, а не замінити його повністю?

Для цього існує `super()`.

### Передбачення

Що виведе цей код?

In [ ]:
class Worker:
    def work(self):
        print("Робочий виконує завдання.")

class Manager(Worker):
    def work(self):
        # Спочатку власна поведінка менеджера
        print("Менеджер координує команду.")
        # Потім делегуємо до наступного класу в MRO
        super().work()

m = Manager()
m.work()

**Пояснення:**

Вивід:
```
Менеджер координує команду.
Робочий виконує завдання.
```

Більшість скаже: "`super()` викликає батьківський клас". У простому одиночному наслідуванні це правда.
Але це **небезпечно неповне визначення**.

> **Точне визначення `super()`:**  
> `super()` делегує виклик **наступному класу в списку MRO** поточного екземпляра.

У простій ієрархії наступним є батько. Але при множинному наслідуванні — починається справжня магія.

### `__init__` та `super()`

Найчастіше `super()` використовується в `__init__`, щоб ініціалізувати батьківські атрибути:

In [ ]:
class Animal:
    def __init__(self, name, sound):
        self.name = name
        self.sound = sound
        print(f"Animal.__init__: ім'я={name}")

    def speak(self):
        print(f"{self.name} каже: {self.sound}!")


class Dog(Animal):
    def __init__(self, name, breed):
        # Викликаємо ініціалізатор батька через super()
        # Передаємо йому аргументи, які він очікує
        super().__init__(name, sound="Гав")
        # Додаємо власний атрибут нащадка
        self.breed = breed
        print(f"Dog.__init__: порода={breed}")

    def fetch(self):
        print(f"{self.name} приносить м'яч!")


rex = Dog("Рекс", "Вівчарка")
rex.speak()   # метод успадкований від Animal
rex.fetch()   # метод власний для Dog
print("Порода:", rex.breed)
print("Звук:", rex.sound)

---

## Частина 3: Множинне наслідування та Проблема Ромба (Diamond Problem)

Python підтримує множинне наслідування — коли клас має більше одного батька.

```
         Base
        /    \
       B      C
        \    /
          D
```

Ця схема нагадує ромб. Якщо `B` і `C` обидва перевизначають метод з `Base` — який з них обере Python для `D`?
Якби Python просто копіював код, `D` отримав би дві конфліктуючі версії методу.

### Передбачення

Якщо `class D(B, C)`, який `ping()` буде викликано? Чому саме цей?

In [ ]:
class Base:
    def ping(self):
        print("ping з Base")

class B(Base):
    def ping(self):
        print("ping з B")

class C(Base):
    def ping(self):
        print("ping з C")

# D успадковує обидва класи — порядок (B, C) має значення!
class D(B, C):
    pass

d = D()
d.ping()

# Подивимось на лінійний порядок пошуку
print("\nMRO для D:")
for i, cls in enumerate(D.mro()):
    print(f"  {i+1}. {cls.__name__}")

**Пояснення — Алгоритм C3 Linearization:**

Python будує MRO за алгоритмом **C3-лінеаризації**. Він гарантує:

| Правило | Що означає |
|---------|------------|
| **Нащадок раніше батька** | `D` завжди перевіряється до `B`, `C`, `Base` |
| **Зберігається порядок зліва направо** | `class D(B, C)` → `B` завжди перед `C` |
| **Без дублювання** | `Base` та `object` з'являються лише один раз, в кінці |

Результат: `[D, B, C, Base, object]`

Python знаходить `ping` у `B` (перший після `D` у списку) і зупиняється.

> **Атрибут `__mro__` та метод `mro()`**  
> `ClassName.__mro__` — кортеж класів (швидкий доступ)  
> `ClassName.mro()` — список (зручніше читати)

---

## Частина 4: Головний WTF — Кооперативне множинне наслідування

Це найважливіша і найбільш дивна частина уроку.

### Передбачення

Перед вами 4 класи: `A`, `B`, `C`, `D`. Всі перевизначають метод `say()` і викликають `super().say()`.

**Напишіть послідовність літер, яку виведе `D().say()`, перш ніж запускати код!**

In [ ]:
class A:
    def say(self):
        print("A")

class B(A):
    def say(self):
        print("B")
        super().say()  # "наступний після B у MRO"

class C(A):
    def say(self):
        print("C")
        super().say()  # "наступний після C у MRO"

class D(B, C):
    def say(self):
        print("D")
        super().say()  # "наступний після D у MRO"

# Спочатку подивимось на MRO
print("MRO:", [cls.__name__ for cls in D.mro()])
print("-" * 30)

# Тепер запускаємо і перевіряємо передбачення
D().say()

**Пояснення шоку:**

Вивід: `D → B → C → A`

Зачекайте. Клас `B` успадковує **тільки `A`**. Але коли `B` викликав `super().say()`, він надрукував `C`!
Як метод `B` зміг викликати метод `C`, з яким не має жодних родинних зв'язків?!

**Ось чому важливо розуміти MRO:**

```
MRO екземпляра D: [D, B, C, A, object]
                    ↑  ↑  ↑  ↑
                    1  2  3  4   (порядок пошуку)
```

`super()` означає: *"знайди поточний клас у MRO екземпляра і передай виклик НАСТУПНОМУ після нього"*.

| Ми знаходимось у... | Поточний MRO | Наступний клас → викликає |
|---------------------|--------------|---------------------------|
| `D` | `[D, B, C, A, object]` | `B` |
| `B` | `[D, B, C, A, object]` | `C` ← **сюрприз!** |
| `C` | `[D, B, C, A, object]` | `A` |
| `A` | `[D, B, C, A, object]` | `object` (базовий, без `say`) |

Цей механізм називається **кооперативне множинне наслідування** (cooperative multiple inheritance).  
Кожен клас "кооперується" і передає естафету далі, гарантуючи, що кожен метод в ієрархії буде викликаний рівно один раз.

> **Ключовий висновок:**  
> `super()` не знає і не цікавиться, чий це батько. Він просто дивиться на MRO **конкретного екземпляра** і знаходить наступний клас у черзі.

---

## Частина 5: Пастки множинного наслідування

Кооперативне наслідування — потужний, але крихкий механізм. Ось дві головні пастки.

### Пастка 1: Обрив ланцюга (The Broken Chain)

Якщо **будь-який** клас у MRO не викликає `super()`, всі класи після нього ніколи не виконаються.
Це механізм "все або нічого".

#### Завдання на дебагінг

У коді нижче конструктор класу `A` ніколи не виконується. Знайдіть причину і виправте.

In [ ]:
class A:
    def __init__(self):
        print("Ініціалізація A")

class Mixin:
    def __init__(self):
        print("Ініціалізація Mixin")
        # ПОМИЛКА ТУТ: ланцюг MRO обривається!
        # Потрібно додати super().__init__()

class B(Mixin, A):
    def __init__(self):
        print("Ініціалізація B")
        super().__init__()  # передаємо далі по MRO

print("MRO для B:", [cls.__name__ for cls in B.mro()])
print("---")
b = B()
print("--- А де 'Ініціалізація A'? ---")

In [ ]:
# ВИПРАВЛЕНА ВЕРСІЯ:
class A:
    def __init__(self):
        print("Ініціалізація A")

class MixinFixed:
    def __init__(self):
        print("Ініціалізація MixinFixed")
        super().__init__()  # тепер передаємо естафету далі!

class BFixed(MixinFixed, A):
    def __init__(self):
        print("Ініціалізація BFixed")
        super().__init__()

print("MRO:", [cls.__name__ for cls in BFixed.mro()])
print("---")
b = BFixed()  # тепер всі три __init__ виконаються

### Пастка 2: Несумісні аргументи (Heterogeneous Signatures)

Оскільки `super()` може викликати будь-який клас у MRO (включно з "несусідніми"), всі методи в ланцюжку повинні мати **сумісні сигнатури**.

Найбезпечніший спосіб — використовувати `*args, **kwargs` для передачі аргументів далі по ланцюгу:

In [ ]:
# БЕЗПЕЧНИЙ ШАБЛОН для кооперативного __init__
# Кожен клас "забирає" свої аргументи і передає решту далі

class Base:
    def __init__(self, **kwargs):
        # object.__init__() не приймає зайвих kwargs,
        # тому на кінці ланцюга словник має бути порожній
        super().__init__(**kwargs)
        print("Base ініціалізовано")

class LogMixin:
    def __init__(self, log_level="INFO", **kwargs):
        # "Забираємо" свій аргумент, решту передаємо далі
        self.log_level = log_level
        super().__init__(**kwargs)
        print(f"LogMixin: рівень логування = {log_level}")

class AuthMixin:
    def __init__(self, role="user", **kwargs):
        self.role = role
        super().__init__(**kwargs)
        print(f"AuthMixin: роль = {role}")

class Service(LogMixin, AuthMixin, Base):
    def __init__(self, name, **kwargs):
        self.name = name
        super().__init__(**kwargs)
        print(f"Service: ім'я = {name}")

print("MRO:", [cls.__name__ for cls in Service.mro()])
print("---")
svc = Service(name="API", log_level="DEBUG", role="admin")
print("---")
print(f"Результат: name={svc.name}, log={svc.log_level}, role={svc.role}")

---

## Частина 6: Наслідування (Is-a) vs Композиція (Has-a)

Тепер, коли ви бачите, наскільки складним стає пошук методів, виникає питання:  
**Чи завжди потрібне наслідування?**

### Чому глибоке наслідування — це проблема

Зміна в базовому класі поширюється хвилею на всі підкласи. Це називається **ефект хвилі (ripple effect)**:
- Ви змінили щось у `Base` — зламали 10 підкласів
- Підклас спирався на неочевидну внутрішню деталь батька
- Тестувати доводиться все дерево цілком

### Два фундаментальних відношення між класами

| Відношення | Синтаксис | Приклад | Коли використовувати |
|------------|-----------|---------|----------------------|
| **Is-a** ("є екземпляром") | `class Dog(Animal)` | Пес — це Тварина | Нащадок **повністю** є різновидом батька |
| **Has-a** ("містить") | `self.engine = Engine()` | Авто **має** двигун | Об'єкт **використовує** інший об'єкт |

> **Правило "Банди чотирьох":**  
> *"Надавайте перевагу композиції об'єктів над наслідуванням класів"*

### Завдання (рефакторинг)

Програміст створив `Car`, успадкувавши його від `Engine`. Але автомобіль — це не двигун!
Перепишіть код, використовуючи **композицію**.

In [ ]:
# ПОГАНО: наслідування (is-a), але Car — НЕ є Engine
class Engine:
    def start(self):
        print("Двигун запущено: Vroooom!")

    def stop(self):
        print("Двигун зупинено.")

class BadCar(Engine):  # Автомобіль успадковує двигун — це неправильно!
    def drive(self):
        self.start()   # викликає метод «батька» Engine
        print("Їдемо!")

bad_car = BadCar()
bad_car.drive()
# Проблема: bad_car.stop() — автомобіль «є» двигуном за цим дизайном!
print("Чи є bad_car двигуном?", isinstance(bad_car, Engine))  # True — це дизайн-помилка

In [ ]:
# ДОБРЕ: композиція (has-a) — автомобіль МАЄ двигун
class Engine:
    def start(self):
        print("Двигун запущено: Vroooom!")

    def stop(self):
        print("Двигун зупинено.")

class GoodCar:
    def __init__(self, engine: Engine):
        # Зберігаємо двигун як атрибут (has-a відношення)
        self.engine = engine

    def drive(self):
        self.engine.start()  # делегуємо двигуну його роботу
        print("Їдемо!")

    def park(self):
        self.engine.stop()
        print("Запарковано.")


# Переваги композиції: двигун можна замінити без зміни Car!
diesel = Engine()
car = GoodCar(engine=diesel)

car.drive()
car.park()

print("Чи є car двигуном?", isinstance(car, Engine))  # False — правильно!

**Переваги композиції:**

| Критерій | Наслідування | Композиція |
|----------|-------------|------------|
| Зв'язування | Жорстке (tight coupling) | Слабке (loose coupling) |
| Зміна реалізації | Може зламати підкласи | Ізольована, безпечна |
| Тестування | Весь ланцюг разом | Компоненти окремо |
| Замінюваність | Складна | Легка (передаємо інший об'єкт) |
| Читабельність | Погіршується з глибиною | Залишається чіткою |

### Коли все ж використовувати наслідування?

Наслідування виправдане, коли виконуються **всі** умови:
1. Нащадок справді є різновидом батька (тест Лісков: `Dog` можна використовувати скрізь, де `Animal`)
2. Нащадок **розширює**, а не **обмежує** поведінку батька
3. Ієрархія не глибша 2-3 рівнів

---

## Частина 7: Чому наслідування порушує інкапсуляцію

В ООП є парадокс: хоча **інкапсуляція** і **наслідування** — два фундаментальних принципи,
**наслідування суттєво послаблює інкапсуляцію**.

### Інкапсуляція: ідея «чорного ящика»

Інкапсуляція означає, що зовнішній код взаємодіє з об'єктом **тільки через публічний інтерфейс** і не знає про деталі реалізації всередині.

Але коли клас `Dog` наслідує `Animal`:
- `Dog` знає і залежить від **внутрішньої реалізації** `Animal`
- Зміна `Animal` може зламати `Dog` навіть якщо `Dog` не змінювали
- `Animal` і `Dog` стають **жорстко зв'язаними** (tight coupling)

Це називається **ефект хвилі (ripple effect)**:

In [ ]:
# Демонстрація ефекту хвилі

class AnimalV1:
    def __init__(self, name):
        self.name = name

    def describe(self):
        # Початкова реалізація: просто виводимо ім'я
        return f"Тварина: {self.name}"


class DogV1(AnimalV1):
    def describe(self):
        # Dog покладається на ВНУТРІШНЮ реалізацію Animal
        base_desc = super().describe()
        return f"{base_desc} (собака)"


dog = DogV1("Рекс")
print(dog.describe())  # Тварина: Рекс (собака)

# --- ТЕПЕР ЗМІНЮЄМО Animal (начебто нешкідливо) ---

class AnimalV2:
    def __init__(self, name, species):
        self.name = name
        self.species = species  # додали новий обов'язковий параметр!

    def describe(self):
        return f"{self.species}: {self.name}"


class DogV2(AnimalV2):
    # Dog НЕ змінювали — але він зламався через зміну Animal!
    def describe(self):
        base_desc = super().describe()
        return f"{base_desc} (собака)"


try:
    dog2 = DogV2("Рекс")  # TypeError: бракує аргументу 'species'
except TypeError as e:
    print(f"Помилка після зміни батька: {e}")
    print("Dog зламався, хоча ми його не чіпали!")

**Той самий приклад через композицію — стійкий до змін:**

In [ ]:
# Та сама зміна в AnimalInfo — але Dog залишається цілим

class AnimalInfo:
    def __init__(self, name, species):
        self.name = name
        self.species = species

    def describe(self):
        return f"{self.species}: {self.name}"


class DogComposition:
    def __init__(self, name, species="Canis lupus"):
        # Dog контролює, як саме він ініціалізує AnimalInfo
        self._info = AnimalInfo(name, species)

    def describe(self):
        # Явно делегуємо те, що нам потрібно
        return f"{self._info.describe()} (собака)"


# AnimalInfo змінила сигнатуру — DogComposition не зламався,
# бо він сам керує своїм атрибутом _info
dog3 = DogComposition("Рекс")
print(dog3.describe())

---

## Підсумок уроку

| # | Ідея | Ключовий факт |
|---|------|---------------|
| 1 | **Наслідування — це пошук, не копіювання** | Python будує MRO і рухається по ньому зліва направо |
| 2 | **MRO завжди можна перевірити** | `ClassName.mro()` або `ClassName.__mro__` |
| 3 | **`super()` — не "батько"** | Це **наступний клас у MRO** поточного екземпляра |
| 4 | **Множинне наслідування вимагає кооперації** | Кожен клас повинен викликати `super()`, інакше ланцюг обривається |
| 5 | **Наслідування порушує інкапсуляцію** | Зміна базового класу може зламати підкласи без жодних змін у них |
| 6 | **Надавайте перевагу композиції** | "has-a" дає слабке зв'язування, ізоляцію та замінюваність компонентів |

---

### Коли що використовувати

```
Питання: "Клас A і клас B — яке між ними відношення?"

Якщо A є різновидом B → Наслідування (is-a)
    Dog is an Animal    → class Dog(Animal)

Якщо A використовує B → Композиція (has-a)
    Car has an Engine   → self.engine = Engine()

Якщо A потребує додаткової поведінки → Mixin (одна вузька функція)
    Service + Logging   → class Service(LogMixin, Base)
```

---

### Що далі?

- **Урок 17:** Поліморфізм. Інкапсуляція. Дандер-методи.